In [1]:
import chromadb
from chromadb.utils import embedding_functions
import sys
sys.path.append('../')

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client     = chromadb.PersistentClient(path="../data/vectorstore")
collection = client.get_or_create_collection(
    name="spotify_metadata",
    embedding_function=ef
)

metadata_docs = [
    {"id": "get_top_genres",
     "text": "get_top_genres returns top music genres ranked by average popularity score. Use when user asks about popular genres, best genres, which genre performs best, genre rankings."},
    {"id": "get_top_artists",
     "text": "get_top_artists returns artists ranked by average popularity. Use when user asks about popular artists, best performers, top musicians, artist rankings."},
    {"id": "find_similar_tracks",
     "text": "find_similar_tracks finds songs similar to a given track using ML cosine similarity on audio features. Use when user asks for recommendations, similar songs, tracks like X, music like this song."},
    {"id": "get_genre_mood_profile",
     "text": "get_genre_mood_profile returns audio fingerprint and mood analysis of a genre including energy, valence, danceability and mood label. Use when user asks about genre characteristics, what does X genre sound like, mood of a genre."},
    {"id": "predict_popularity",
     "text": "predict_popularity uses a neural network to predict how popular a track will be based on its audio features. Use when user asks about predicting popularity, how well will this song do, popularity forecast."},
    {"id": "simulate_playlist",
     "text": "simulate_playlist runs Monte Carlo simulation to predict listener engagement and drop-off rates across a playlist. Use when user asks about playlist quality, engagement, will listeners stay, playlist analysis."},
]

collection.upsert(
    ids=[d["id"] for d in metadata_docs],
    documents=[d["text"] for d in metadata_docs]
)

print(f"✅ Vector store created with {collection.count()} documents")

/Users/user/Documents/spotify_intelligence_copilot/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6853.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store created with 6 documents


In [2]:
def retrieve_tools(query, n=3):
    results = collection.query(query_texts=[query], n_results=n)
    return results['ids'][0], results['documents'][0]

test_queries = [
    "Which genres are most popular?",
    "Find me songs similar to Shape of You",
    "What is the mood of jazz music?",
    "Will this track be popular?",
    "Analyze my playlist engagement",
]

for q in test_queries:
    ids, _ = retrieve_tools(q)
    print(f"🔍 '{q}'")
    print(f"   → {ids}\n")

🔍 'Which genres are most popular?'
   → ['get_top_genres', 'get_top_artists', 'get_genre_mood_profile']

🔍 'Find me songs similar to Shape of You'
   → ['find_similar_tracks', 'get_genre_mood_profile', 'predict_popularity']

🔍 'What is the mood of jazz music?'
   → ['get_genre_mood_profile', 'get_top_genres', 'predict_popularity']

🔍 'Will this track be popular?'
   → ['predict_popularity', 'simulate_playlist', 'find_similar_tracks']

🔍 'Analyze my playlist engagement'
   → ['simulate_playlist', 'get_genre_mood_profile', 'get_top_genres']



In [3]:
rag_code = '''
import chromadb
from chromadb.utils import embedding_functions
import os

BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
_client = chromadb.PersistentClient(path=os.path.join(BASE, "data", "vectorstore"))
_collection = _client.get_or_create_collection(
    name="spotify_metadata",
    embedding_function=_ef
)

def retrieve_relevant_tools(query, n_results=3):
    results = _collection.query(query_texts=[query], n_results=n_results)
    return results["ids"][0], results["documents"][0]
'''

with open('../src/rag.py', 'w') as f:
    f.write(rag_code)

print("✅ src/rag.py created!")

✅ src/rag.py created!
